In [2]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path(r"C:\Users\tyiho\DSE3101 Project\dse3101investmentproject\Datasets\13F_clean_files")

all_quarters = []

# loop through all parquet files
for file in DATA_DIR.glob("*.parquet"):
    print("Processing:", file.name)

    df = pd.read_parquet(file)

    # keep latest filing per institution per quarter
    latest = (
        df.sort_values("FILING_DATE")
        .groupby(["CIK", "PERIODOFREPORT"], as_index=False)
        .last()
    )

    all_quarters.append(latest)

# combine all quarters
df_all = pd.concat(all_quarters, ignore_index=True)

Processing: 01dec2024-28feb2025_clean_df.parquet
Processing: 01jan2024-29feb2024_clean_df.parquet
Processing: 01jun2024-31aug2024_clean_df.parquet
Processing: 01jun2025-31aug2025_clean_df.parquet
Processing: 01mar2024-31may2024_clean_df.parquet
Processing: 01mar2025-31may2025_clean_df.parquet
Processing: 01sep2024-30nov2024_clean_df.parquet
Processing: 01sep2025-30nov2025_clean_df.parquet
Processing: 2013q2_clean_df.parquet
Processing: 2013q3_clean_df.parquet
Processing: 2013q4_clean_df.parquet
Processing: 2014q1_clean_df.parquet
Processing: 2014q2_clean_df.parquet
Processing: 2014q3_clean_df.parquet
Processing: 2014q4_clean_df.parquet
Processing: 2015q1_clean_df.parquet
Processing: 2015q2_clean_df.parquet
Processing: 2015q3_clean_df.parquet
Processing: 2015q4_clean_df.parquet
Processing: 2016q1_clean_df.parquet
Processing: 2016q2_clean_df.parquet
Processing: 2016q3_clean_df.parquet
Processing: 2016q4_clean_df.parquet
Processing: 2017q1_clean_df.parquet
Processing: 2017q2_clean_df.parq

In [5]:
institutions_per_quarter = (
    df_all.groupby("PERIODOFREPORT")
    .agg(num_institutions=("CIK", "nunique"))
    .reset_index()
)

print(institutions_per_quarter)
institutions_per_quarter.to_csv("institutions_per_quarter.csv", index=False)

   PERIODOFREPORT  num_institutions
0      2013-03-31                43
1      2013-06-30              2189
2      2013-09-30              2202
3      2013-12-31              2450
4      2014-03-31              2452
5      2014-06-30              2472
6      2014-09-30              2467
7      2014-12-31              2691
8      2015-03-31              2685
9      2015-06-30              2686
10     2015-09-30              2709
11     2015-12-31              2808
12     2016-03-31              2839
13     2016-06-30              2827
14     2016-09-30              2815
15     2016-12-31              2965
16     2017-03-31              2949
17     2017-06-30              2948
18     2017-09-30              2956
19     2017-12-31              3236
20     2018-03-31              3263
21     2018-06-30              3286
22     2018-09-30              3280
23     2018-12-31              3578
24     2019-03-31              3561
25     2019-06-30              3563
26     2019-09-30           

In [3]:
institution_values = df_all[
    ["PERIODOFREPORT", "CIK", "FILINGMANAGER_NAME", "TABLEVALUETOTAL", "FILING_DATE"]
].copy()

print(institution_values.head())
institution_values.to_csv("institution_values.csv", index=False)

  PERIODOFREPORT         CIK                   FILINGMANAGER_NAME  \
0     2024-12-31  0000002230  ADAMS DIVERSIFIED EQUITY FUND, INC.   
1     2024-12-31  0000004977                            Aflac Inc   
2     2024-12-31  0000005272   AMERICAN INTERNATIONAL GROUP, INC.   
3     2024-12-31  0000007195       ARGUS INVESTORS' COUNSEL, INC.   
4     2024-12-31  0000007773           ASSET PLANNING CORPORATION   

   TABLEVALUETOTAL FILING_DATE  
0     2.642992e+09  2025-02-11  
1     1.765024e+08  2025-02-05  
2     4.141015e+09  2025-02-11  
3     1.497620e+08  2025-01-27  
4     1.597002e+08  2025-01-24  


In [4]:
import pandas as pd

df = institution_values.copy()

# convert date
df["PERIODOFREPORT"] = pd.to_datetime(df["PERIODOFREPORT"])

# remove abnormal quarter
df = df[df["PERIODOFREPORT"] != "2025-06-30"]   # assuming 2025Q2

# ---- 1. Institutions active in latest quarter ----
latest_q = df["PERIODOFREPORT"].max()

active_ciks = df[df["PERIODOFREPORT"] == latest_q]["CIK"].unique()

df = df[df["CIK"].isin(active_ciks)]

# ---- 2. Institutions with at least 20 quarters of filings ----
filing_counts = df.groupby("CIK").size()

valid_ciks = filing_counts[filing_counts >= 20].index

df = df[df["CIK"].isin(valid_ciks)]

# ---- 3. Institutions with avg AUM >= 100M ----
aum_avg = df.groupby("CIK")["TABLEVALUETOTAL"].mean()

large_ciks = aum_avg[aum_avg >= 100_000].index   # if values are in thousands
# use 100_000_000 if raw dollars

df_filtered = df[df["CIK"].isin(large_ciks)]

In [6]:
df_filtered.to_csv("filtered_institution_values.csv", index=False)

In [ ]:
# number of unique institutions befpre and after filtering
print(institution_values["CIK"].nunique())
df_filtered["CIK"].nunique()

9763


3146

In [ ]:
#average quarters per institution
df_filtered.groupby("CIK").size().describe()

count    3146.000000
mean       37.238716
std        11.227517
min        20.000000
25%        27.000000
50%        37.000000
75%        50.000000
max        52.000000
dtype: float64